In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

df = pd.read_csv("../data/processed/features_2022.csv")
df["flooded"] = (df.flood_frac >= 0.10).astype(int)

# rain features hata do: ek hi event me ye har cell pe same hain
X = df[["elevation", "slope", "dist_water_m", "lat", "lon"]]
y = df["flooded"]

# spatial split: lat ke hisaab se, taaki paas ke cells train/test me mix na ho
cut = df.lat.quantile(0.75)
train = df.lat <= cut
Xtr, Xte, ytr, yte = X[train], X[~train], y[train], y[~train]

model = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    scale_pos_weight=(ytr == 0).sum() / max((ytr == 1).sum(), 1),
    eval_metric="logloss",
)
model.fit(Xtr, ytr)

p = model.predict_proba(Xte)[:, 1]
print("Test ROC-AUC:", round(roc_auc_score(yte, p), 3))
print(classification_report(yte, (p > 0.5).astype(int)))
print(dict(zip(X.columns, model.feature_importances_.round(3))))

Test ROC-AUC: 0.703
              precision    recall  f1-score   support

           0       0.82      0.92      0.87      1063
           1       0.51      0.29      0.37       297

    accuracy                           0.78      1360
   macro avg       0.67      0.60      0.62      1360
weighted avg       0.75      0.78      0.76      1360

{'elevation': np.float32(0.125), 'slope': np.float32(0.123), 'dist_water_m': np.float32(0.128), 'lat': np.float32(0.249), 'lon': np.float32(0.376)}


In [9]:
for thr in [0.3, 0.4, 0.5]:
    pred = (p > thr).astype(int)
    print("Decision threshold", thr)
    print(classification_report(yte, pred, digits=2))

print(dict(zip(X.columns, model.feature_importances_.round(3))))

Decision threshold 0.3
              precision    recall  f1-score   support

           0       0.85      0.83      0.84      1063
           1       0.43      0.48      0.46       297

    accuracy                           0.75      1360
   macro avg       0.64      0.65      0.65      1360
weighted avg       0.76      0.75      0.75      1360

Decision threshold 0.4
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1063
           1       0.48      0.40      0.44       297

    accuracy                           0.77      1360
   macro avg       0.66      0.64      0.65      1360
weighted avg       0.76      0.77      0.77      1360

Decision threshold 0.5
              precision    recall  f1-score   support

           0       0.82      0.92      0.87      1063
           1       0.51      0.29      0.37       297

    accuracy                           0.78      1360
   macro avg       0.67      0.60      0.62      1360
weigh

In [10]:
from sklearn.metrics import roc_auc_score, classification_report

feats = ["elevation", "slope", "dist_water_m"]   # lat, lon hata diye
Xtr2, Xte2 = df.loc[train, feats], df.loc[~train, feats]

model2 = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    scale_pos_weight=(ytr == 0).sum() / max((ytr == 1).sum(), 1),
    eval_metric="logloss",
)
model2.fit(Xtr2, ytr)
p2 = model2.predict_proba(Xte2)[:, 1]

print("ROC-AUC (no lat/lon):", round(roc_auc_score(yte, p2), 3))
print(classification_report(yte, (p2 > 0.3).astype(int), digits=2))
print(dict(zip(feats, model2.feature_importances_.round(3))))

ROC-AUC (no lat/lon): 0.492
              precision    recall  f1-score   support

           0       0.76      0.51      0.61      1063
           1       0.20      0.44      0.27       297

    accuracy                           0.49      1360
   macro avg       0.48      0.47      0.44      1360
weighted avg       0.64      0.49      0.54      1360

{'elevation': np.float32(0.492), 'slope': np.float32(0.225), 'dist_water_m': np.float32(0.283)}


In [11]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

# 1. har feature akela: train vs test AUC
for f in ["elevation", "slope", "dist_water_m"]:
    print(f, "train AUC:", round(roc_auc_score(ytr, df.loc[train, f]), 3),
             "test AUC:", round(roc_auc_score(yte, df.loc[~train, f]), 3))

# 2. flood rate lat band ke hisaab se
df["lat_band"] = pd.cut(df.lat, 7)
print(df.groupby("lat_band", observed=True).flooded.mean().round(3))

# 3. dist_water ki range
print(df.dist_water_m.describe())

# 4. blocked cross-validation (0.1 degree ke blocks), zyada fair
df["block"] = (df.lat // 0.1).astype(int) * 100 + (df.lon // 0.1).astype(int)
feats = ["elevation", "slope", "dist_water_m"]
aucs = []
for tr, te in GroupKFold(5).split(df, df.flooded, df.block):
    m = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05)
    m.fit(df.iloc[tr][feats], df.flooded.iloc[tr])
    aucs.append(roc_auc_score(df.flooded.iloc[te], m.predict_proba(df.iloc[te][feats])[:, 1]))
print("Blocked CV AUC:", np.round(aucs, 3), "mean", round(np.mean(aucs), 3))

elevation train AUC: 0.637 test AUC: 0.33
slope train AUC: 0.643 test AUC: 0.544
dist_water_m train AUC: 0.605 test AUC: 0.565
lat_band
(20.204, 20.304]    0.039
(20.304, 20.402]    0.184
(20.402, 20.501]    0.026
(20.501, 20.599]    0.036
(20.599, 20.698]    0.148
(20.698, 20.796]    0.149
(20.796, 20.895]    0.254
Name: flooded, dtype: float64
count     5600.000000
mean      3001.734694
std       3344.952676
min          0.000000
25%        336.308518
50%       1905.467528
75%       4471.720423
max      15983.546857
Name: dist_water_m, dtype: float64
Blocked CV AUC: [0.759 0.647 0.661 0.63  0.715] mean 0.682


In [12]:
# flood patches jo bahut unchi ya door-from-river zameen par hain, unhe dekho
print(df.groupby("flooded")[["elevation","slope","dist_water_m"]].median().round(2))

for thr in [0.10, 0.20, 0.30]:
    y_t = (df.flood_frac >= thr).astype(int)
    aucs = []
    for tr, te in GroupKFold(5).split(df, y_t, df.block):
        m = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05)
        m.fit(df.iloc[tr][feats], y_t.iloc[tr])
        aucs.append(roc_auc_score(y_t.iloc[te], m.predict_proba(df.iloc[te][feats])[:, 1]))
    print("label thr", thr, "positives", y_t.sum(), "blocked AUC", round(np.mean(aucs), 3))

         elevation  slope  dist_water_m
flooded                                
0             4.16   0.96       1730.02
1             4.55   1.03       2806.53
label thr 0.1 positives 668 blocked AUC 0.682
label thr 0.2 positives 380 blocked AUC 0.675
label thr 0.3 positives 250 blocked AUC 0.713


In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

df = pd.read_csv("../data/processed/features_2022_v2.csv")
df["block"] = (df.lat // 0.1).astype(int) * 100 + (df.lon // 0.1).astype(int)

sets = {
    "old (3 feats)": ["elevation", "slope", "dist_water_m"],
    "+ HAND/neighbourhood": ["elevation", "slope", "dist_water_m", "hand", "elev_mean_1km", "elev_rel"],
    "+ land cover": ["elevation", "slope", "dist_water_m", "hand", "elev_mean_1km", "elev_rel",
                     "cropland", "builtup", "wetland_mangrove"],
}

for thr in [0.10, 0.30]:
    y = (df.flood_frac >= thr).astype(int)
    for name, fs in sets.items():
        aucs = []
        for tr, te in GroupKFold(5).split(df, y, df.block):
            m = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05)
            m.fit(df.iloc[tr][fs], y.iloc[tr])
            aucs.append(roc_auc_score(y.iloc[te], m.predict_proba(df.iloc[te][fs])[:, 1]))
        print(f"thr {thr} | {name:22s} | blocked AUC {np.mean(aucs):.3f}")

print(df.groupby(df.flood_frac >= 0.10)[["cropland", "hand", "elev_rel"]].mean().round(3))

thr 0.1 | old (3 feats)          | blocked AUC 0.682
thr 0.1 | + HAND/neighbourhood   | blocked AUC 0.733
thr 0.1 | + land cover           | blocked AUC 0.760
thr 0.3 | old (3 feats)          | blocked AUC 0.713
thr 0.3 | + HAND/neighbourhood   | blocked AUC 0.758
thr 0.3 | + land cover           | blocked AUC 0.780
            cropland   hand  elev_rel
flood_frac                           
False          0.505  1.304     0.020
True           0.769  0.777    -0.145
